In [1]:
#CELL 1 - IMPORT REQUIRED LIBRARIES

import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
#CELL 2 - CONFIGURATION

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

MASTER_FILE = Path("../01_Master_Datasets/Customer_Reviews.csv")
OUTPUT_FILE = Path("Customer_Reviews_Dirty.csv")

print("=" * 60)
print("Customer Reviews Dirty Dataset Generation")
print("=" * 60)
print(f"Random Seed : {RANDOM_SEED}")
print(f"Master File : {MASTER_FILE}")
print(f"Output File : {OUTPUT_FILE}")

Customer Reviews Dirty Dataset Generation
Random Seed : 42
Master File : ..\01_Master_Datasets\Customer_Reviews.csv
Output File : Customer_Reviews_Dirty.csv


In [5]:
#CELL 3 - LOAD MASTER DATASET 

reviews_master = pd.read_csv(MASTER_FILE)

print("=" * 60)
print("MASTER DATASET LOADED")
print("=" * 60)

print(f"Rows    : {reviews_master.shape[0]}")
print(f"Columns : {reviews_master.shape[1]}")

display(reviews_master.head())

MASTER DATASET LOADED
Rows    : 1156
Columns : 6


,Review_ID,Order_ID,Customer_ID,Product_ID,Review_Date,Rating
0,R000001,O000773,C0099,P0021,2025-12-02,3
1,R000002,O000534,C0086,P0061,2024-10-08,5
2,R000003,O001706,C0365,P0180,2023-03-28,5
3,R000004,O000932,C0374,P0020,2025-10-24,1
4,R000005,O002310,C0279,P0048,2024-12-31,5


In [6]:
#CELL 4 - CREATE WORKING COPY

reviews_dirty = reviews_master.copy(deep=True)

print("=" * 60)
print("WORKING COPY CREATED")
print("=" * 60)

print("Master Dataset ID :", id(reviews_master))
print("Working Copy ID   :", id(reviews_dirty))

assert id(reviews_master) != id(reviews_dirty)

print("\nSUCCESS: Master dataset remains protected.")

WORKING COPY CREATED
Master Dataset ID : 1385145080416
Working Copy ID   : 1387223306064

SUCCESS: Master dataset remains protected.


In [7]:
#CELL 5 - INITIAL DATASET VALIDATION

print("=" * 60)
print("INITIAL DATASET VALIDATION")
print("=" * 60)

print(f"Rows               : {reviews_dirty.shape[0]}")
print(f"Columns            : {reviews_dirty.shape[1]}")
print(f"Missing Values     : {reviews_dirty.isnull().sum().sum()}")
print(f"Duplicate Records  : {reviews_dirty.duplicated().sum()}")

assert reviews_dirty.shape == reviews_master.shape
assert reviews_dirty.isnull().sum().sum() == 0
assert reviews_dirty.duplicated().sum() == 0

print("\nSUCCESS: Initial dataset validation passed.")

INITIAL DATASET VALIDATION
Rows               : 1156
Columns            : 6
Missing Values     : 0
Duplicate Records  : 0

SUCCESS: Initial dataset validation passed.


In [8]:
#CELL 6 - INITIALIZE MODIFICATION TRACKER

modified_rows = set()

print("=" * 60)
print("MODIFICATION TRACKER INITIALIZED")
print("=" * 60)

print(f"Modified Rows : {len(modified_rows)}")

assert len(modified_rows) == 0

print("\nSUCCESS: Modification tracker initialized.")

MODIFICATION TRACKER INITIALIZED
Modified Rows : 0

SUCCESS: Modification tracker initialized.


In [9]:
#CELL 7 - BLANK Review_Date IMPLEMENTATION

print("=" * 60)
print("ERROR 01 — BLANK REVIEW_DATE")
print("Stage 1 — Implementation")
print("=" * 60)

ERROR_COUNT = 20

available_rows = sorted(
    set(reviews_dirty.index) - modified_rows
)

selected_rows = np.random.choice(
    available_rows,
    size=ERROR_COUNT,
    replace=False
)

selected_rows = sorted(selected_rows)

original_dates = reviews_dirty.loc[
    selected_rows,
    "Review_Date"
].copy()

reviews_dirty.loc[
    selected_rows,
    "Review_Date"
] = np.nan

print(f"Rows Selected : {len(selected_rows)}")
print("Implementation Complete.")

ERROR 01 — BLANK REVIEW_DATE
Stage 1 — Implementation
Rows Selected : 20
Implementation Complete.


In [10]:
#CELL 8 - BLANK Review_Date VALIDATION

print("=" * 60)
print("ERROR 01 — BLANK REVIEW_DATE")
print("Stage 2 — Validation")
print("=" * 60)

actual_blank = reviews_dirty["Review_Date"].isna().sum()

print(f"Expected Blank Dates : {ERROR_COUNT}")
print(f"Actual Blank Dates   : {actual_blank}")

assert actual_blank == ERROR_COUNT

validation_df = pd.DataFrame({
    "Row": selected_rows,
    "Before": original_dates.values,
    "After": reviews_dirty.loc[selected_rows, "Review_Date"].values
})

print("\nFirst Five Changes")
display(validation_df.head())

print("\nComplete Modified Rows")
display(reviews_dirty.loc[selected_rows])

print("\nBefore vs After (repr)")

for i in range(min(5, len(selected_rows))):
    print(
        f"Row {selected_rows[i]} | "
        f"Before = {repr(original_dates.iloc[i])} | "
        f"After = {repr(reviews_dirty.loc[selected_rows[i], 'Review_Date'])}"
    )

print("\nSUCCESS: Blank Review_Date validation passed.")

ERROR 01 — BLANK REVIEW_DATE
Stage 2 — Validation
Expected Blank Dates : 20
Actual Blank Dates   : 20

First Five Changes


,Row,Before,After
0,109,2023-07-19,NaN
1,128,2024-11-11,NaN
2,140,2023-01-30,NaN
3,170,2023-07-25,NaN
4,265,2025-10-21,NaN



Complete Modified Rows


,Review_ID,Order_ID,Customer_ID,Product_ID,Review_Date,Rating
109,R000110,O001141,C0549,P0171,NaN,2
128,R000129,O000599,C0354,P0113,NaN,4
140,R000141,O004096,C0584,P0068,NaN,3
170,R000171,O004713,C0519,P0199,NaN,1
265,R000266,O001045,C0099,P0103,NaN,4
299,R000300,O001855,C0675,P0020,NaN,5
318,R000319,O002613,C0603,P0004,NaN,4
359,R000360,O001916,C0079,P0158,NaN,1
549,R000550,O002486,C0498,P0176,NaN,2
597,R000598,O003633,C0395,P0135,NaN,3



Before vs After (repr)
Row 109 | Before = '2023-07-19' | After = nan
Row 128 | Before = '2024-11-11' | After = nan
Row 140 | Before = '2023-01-30' | After = nan
Row 170 | Before = '2023-07-25' | After = nan
Row 265 | Before = '2025-10-21' | After = nan

SUCCESS: Blank Review_Date validation passed.


In [11]:
# CELL 9 - BLANK Review_Date MODIFICATION TRACKER UPDATE

print("=" * 60)
print("ERROR 01 — BLANK REVIEW_DATE")
print("Stage 3 — Tracker Update")
print("=" * 60)

previous_count = len(modified_rows)

modified_rows.update(selected_rows)

current_count = len(modified_rows)

print(f"Previous Tracker Count : {previous_count}")
print(f"Rows Added             : {len(selected_rows)}")
print(f"Current Tracker Count  : {current_count}")

assert current_count == previous_count + len(selected_rows)

print("\nSUCCESS: Modification tracker updated successfully.")

ERROR 01 — BLANK REVIEW_DATE
Stage 3 — Tracker Update
Previous Tracker Count : 0
Rows Added             : 20
Current Tracker Count  : 20

SUCCESS: Modification tracker updated successfully.


In [12]:
#CELL 10 - BLANK RATING IMPLEMENTATION


print("=" * 60)
print("ERROR 02 — BLANK RATING")
print("Stage 1 — Implementation")
print("=" * 60)

ERROR_COUNT = 20

available_rows = sorted(
    set(reviews_dirty.index) - modified_rows
)

selected_rows = np.random.choice(
    available_rows,
    size=ERROR_COUNT,
    replace=False
)

selected_rows = sorted(selected_rows)

original_ratings = reviews_dirty.loc[
    selected_rows,
    "Rating"
].copy()

# Convert to nullable integer so missing values are supported
reviews_dirty["Rating"] = reviews_dirty["Rating"].astype("Int64")

reviews_dirty.loc[
    selected_rows,
    "Rating"
] = pd.NA

print(f"Rows Selected : {len(selected_rows)}")
print("Implementation Complete.")

ERROR 02 — BLANK RATING
Stage 1 — Implementation
Rows Selected : 20
Implementation Complete.


In [13]:
#CELL 11 - BLANK RATING VALIDATION

print("=" * 60)
print("ERROR 02 — BLANK RATING")
print("Stage 2 — Validation")
print("=" * 60)

actual_blank = reviews_dirty["Rating"].isna().sum()

print(f"Expected Blank Ratings : {ERROR_COUNT}")
print(f"Actual Blank Ratings   : {actual_blank}")

assert actual_blank == ERROR_COUNT

validation_df = pd.DataFrame({
    "Row": selected_rows,
    "Before": original_ratings.values,
    "After": reviews_dirty.loc[selected_rows, "Rating"].values
})

print("\nFirst Five Changes")
display(validation_df.head())

print("\nComplete Modified Rows")
display(reviews_dirty.loc[selected_rows])

print("\nBefore vs After (repr)")

for i in range(min(5, len(selected_rows))):
    print(
        f"Row {selected_rows[i]} | "
        f"Before = {repr(original_ratings.iloc[i])} | "
        f"After = {repr(reviews_dirty.loc[selected_rows[i], 'Rating'])}"
    )

print("\nSUCCESS: Blank Rating validation passed.")

ERROR 02 — BLANK RATING
Stage 2 — Validation
Expected Blank Ratings : 20
Actual Blank Ratings   : 20

First Five Changes


,Row,Before,After
0,57,5,<NA>
1,66,4,<NA>
2,192,5,<NA>
3,263,5,<NA>
4,295,5,<NA>



Complete Modified Rows


,Review_ID,Order_ID,Customer_ID,Product_ID,Review_Date,Rating
57,R000058,O000412,C0435,P0133,2025-02-22,<NA>
66,R000067,O004063,C0289,P0161,2024-09-07,<NA>
192,R000193,O000014,C0503,P0037,2025-03-15,<NA>
263,R000264,O003138,C0476,P0188,2025-06-05,<NA>
295,R000296,O004349,C0302,P0162,2024-09-16,<NA>
304,R000305,O001070,C0555,P0123,2023-09-13,<NA>
543,R000544,O002758,C0097,P0186,2025-01-22,<NA>
568,R000569,O001079,C0039,P0190,2024-10-24,<NA>
586,R000587,O003397,C0593,P0082,2025-09-28,<NA>
684,R000685,O001087,C0380,P0164,2024-07-02,<NA>



Before vs After (repr)
Row 57 | Before = 5 | After = <NA>
Row 66 | Before = 4 | After = <NA>
Row 192 | Before = 5 | After = <NA>
Row 263 | Before = 5 | After = <NA>
Row 295 | Before = 5 | After = <NA>

SUCCESS: Blank Rating validation passed.


In [14]:
#CELL 12 - BLANK RATING TRACKER UPDATE

print("=" * 60)
print("ERROR 02 — BLANK RATING")
print("Stage 3 — Tracker Update")
print("=" * 60)

previous_count = len(modified_rows)

modified_rows.update(selected_rows)

current_count = len(modified_rows)

print(f"Previous Tracker Count : {previous_count}")
print(f"Rows Added             : {len(selected_rows)}")
print(f"Current Tracker Count  : {current_count}")

assert current_count == previous_count + len(selected_rows)

print("\nSUCCESS: Modification tracker updated successfully.")

ERROR 02 — BLANK RATING
Stage 3 — Tracker Update
Previous Tracker Count : 20
Rows Added             : 20
Current Tracker Count  : 40

SUCCESS: Modification tracker updated successfully.


In [15]:
#CELL 13 - DUPLICATE RECORDS

print("=" * 60)
print("ERROR 03 — DUPLICATE RECORDS")
print("Stage 1 — Implementation")
print("=" * 60)

ERROR_COUNT = 25

original_row_count = len(reviews_dirty)

duplicate_rows = np.random.choice(
    reviews_dirty.index,
    size=ERROR_COUNT,
    replace=False
)

duplicate_rows = sorted(duplicate_rows)

duplicate_records = reviews_dirty.loc[
    duplicate_rows
].copy()

reviews_dirty = pd.concat(
    [reviews_dirty, duplicate_records],
    ignore_index=True
)

print(f"Original Rows        : {original_row_count}")
print(f"Duplicate Rows Added : {ERROR_COUNT}")
print(f"Rows After Append    : {len(reviews_dirty)}")

ERROR 03 — DUPLICATE RECORDS
Stage 1 — Implementation
Original Rows        : 1156
Duplicate Rows Added : 25
Rows After Append    : 1181


In [16]:
#CELL 14 - DUPLICATE RECORDS VALIDATION

print("=" * 60)
print("ERROR 03 — DUPLICATE RECORDS")
print("Stage 2 — Validation")
print("=" * 60)

# Shuffle entire dataset
reviews_dirty = reviews_dirty.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

duplicate_count = reviews_dirty.duplicated().sum()

print(f"Expected Final Rows      : {original_row_count + ERROR_COUNT}")
print(f"Actual Final Rows        : {len(reviews_dirty)}")

print(f"\nExpected Duplicate Rows  : {ERROR_COUNT}")
print(f"Actual Duplicate Rows    : {duplicate_count}")

assert len(reviews_dirty) == original_row_count + ERROR_COUNT
assert duplicate_count == ERROR_COUNT

print("\nSample Duplicate Records")

duplicate_sample = reviews_dirty[
    reviews_dirty.duplicated(keep=False)
].sort_values(
    by=["Review_ID"]
)

display(duplicate_sample.head(10))

print("\nSUCCESS: Duplicate validation passed.")

ERROR 03 — DUPLICATE RECORDS
Stage 2 — Validation
Expected Final Rows      : 1181
Actual Final Rows        : 1181

Expected Duplicate Rows  : 25
Actual Duplicate Rows    : 25

Sample Duplicate Records


,Review_ID,Order_ID,Customer_ID,Product_ID,Review_Date,Rating
1072,R000071,O003457,C0253,P0109,2024-05-23,4
48,R000071,O003457,C0253,P0109,2024-05-23,4
705,R000096,O000108,C0063,P0035,2025-02-25,4
1059,R000096,O000108,C0063,P0035,2025-02-25,4
232,R000140,O001954,C0562,P0040,2024-09-12,3
409,R000140,O001954,C0562,P0040,2024-09-12,3
66,R000145,O003557,C0735,P0177,2024-11-05,1
527,R000145,O003557,C0735,P0177,2024-11-05,1
853,R000204,O004808,C0347,P0112,2023-11-23,5
730,R000204,O004808,C0347,P0112,2023-11-23,5



SUCCESS: Duplicate validation passed.


In [17]:
#CELL 15 - DUPLICATE RECORDS TRACKER UPDATE

print("=" * 60)
print("ERROR 03 — DUPLICATE RECORDS")
print("Stage 3 — Tracker Update")
print("=" * 60)

previous_count = len(modified_rows)

current_count = len(modified_rows)

print(f"Previous Tracker Count : {previous_count}")
print("Rows Added             : 0")
print(f"Current Tracker Count  : {current_count}")

assert previous_count == current_count

print("\nSUCCESS: Tracker unchanged (duplicates are appended records, not new row-level modifications).")

ERROR 03 — DUPLICATE RECORDS
Stage 3 — Tracker Update
Previous Tracker Count : 40
Rows Added             : 0
Current Tracker Count  : 40

SUCCESS: Tracker unchanged (duplicates are appended records, not new row-level modifications).


In [18]:
#CELL 16 - FINAL DATASET VALIDATION

print("=" * 60)
print("FINAL DATASET VALIDATION")
print("=" * 60)

print(f"Rows                : {len(reviews_dirty)}")
print(f"Columns             : {reviews_dirty.shape[1]}")
print(f"Duplicate Records   : {reviews_dirty.duplicated().sum()}")
print(f"Blank Review Dates  : {reviews_dirty['Review_Date'].isna().sum()}")
print(f"Blank Ratings       : {reviews_dirty['Rating'].isna().sum()}")

assert len(reviews_dirty) == 1181
assert reviews_dirty.shape[1] == 6
assert reviews_dirty.duplicated().sum() == 25
assert reviews_dirty["Review_Date"].isna().sum() == 20
assert reviews_dirty["Rating"].isna().sum() == 20

print("\nSUCCESS: Final dataset validation passed.")

FINAL DATASET VALIDATION
Rows                : 1181
Columns             : 6
Duplicate Records   : 25
Blank Review Dates  : 20
Blank Ratings       : 20

SUCCESS: Final dataset validation passed.


In [19]:
#CELL 17 - MASTER DATASET PROTECTION VALIDATION

print("=" * 60)
print("MASTER DATASET PROTECTION VALIDATION")
print("=" * 60)

assert len(reviews_master) == 1156
assert reviews_master.shape == (1156, 6)
assert reviews_master.isnull().sum().sum() == 0
assert reviews_master.duplicated().sum() == 0

print("Master Dataset Rows       :", len(reviews_master))
print("Working Dataset Rows      :", len(reviews_dirty))

print("\nSUCCESS: Customer_Reviews.csv remains unchanged.")

MASTER DATASET PROTECTION VALIDATION
Master Dataset Rows       : 1156
Working Dataset Rows      : 1181

SUCCESS: Customer_Reviews.csv remains unchanged.


In [22]:
#CELL 18 - BUSINESS RELATIONSHIP VALIDATION

print("=" * 60)
print("BUSINESS RELATIONSHIP VALIDATION")
print("=" * 60)

customers = pd.read_csv("../01_Master_Datasets/Customer_Master.csv")
products = pd.read_csv("../01_Master_Datasets/Product_Master.csv")
orders = pd.read_csv("../01_Master_Datasets/Orders.csv")

assert (
    reviews_dirty["Order_ID"]
    .dropna()
    .isin(orders["Order_ID"])
    .all()
)

assert (
    reviews_dirty["Customer_ID"]
    .dropna()
    .isin(customers["Customer_ID"])
    .all()
)

assert (
    reviews_dirty["Product_ID"]
    .dropna()
    .isin(products["Product_ID"])
    .all()
)

print("✓ All Order_ID values are valid.")
print("✓ All Customer_ID values are valid.")
print("✓ All Product_ID values are valid.")

print("\nSUCCESS: Business relationships remain intact.")

BUSINESS RELATIONSHIP VALIDATION
✓ All Order_ID values are valid.
✓ All Customer_ID values are valid.
✓ All Product_ID values are valid.

SUCCESS: Business relationships remain intact.


In [23]:
#CELL 19 - ETL AUDIT SUMMARY

print("=" * 60)
print("ETL AUDIT SUMMARY")
print("=" * 60)

audit_summary = pd.DataFrame({
    "Error Type": [
        "Blank Review_Date",
        "Blank Rating",
        "Duplicate Records"
    ],
    "Affected Rows": [
        20,
        20,
        25
    ],
    "Validation Status": [
        "Passed",
        "Passed",
        "Passed"
    ]
})

display(audit_summary)

print(f"Original Dataset Size : {len(reviews_master)}")
print(f"Final Dirty Dataset   : {len(reviews_dirty)}")
print(f"Modified Original Rows: {len(modified_rows)}")
print(f"Duplicate Rows Added  : 25")

print("\nSUCCESS: ETL audit completed successfully.")

ETL AUDIT SUMMARY


,Error Type,Affected Rows,Validation Status
0,Blank Review_Date,20,Passed
1,Blank Rating,20,Passed
2,Duplicate Records,25,Passed


Original Dataset Size : 1156
Final Dirty Dataset   : 1181
Modified Original Rows: 40
Duplicate Rows Added  : 25

SUCCESS: ETL audit completed successfully.


In [24]:
#CELL 20 - EXPORT DIRTY DATASET

reviews_dirty.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 60)
print("EXPORT COMPLETED")
print("=" * 60)

print(f"File Name : {OUTPUT_FILE}")
print(f"Rows      : {len(reviews_dirty)}")
print(f"Columns   : {reviews_dirty.shape[1]}")

print("\nCustomer_Reviews_Dirty.csv exported successfully.")

EXPORT COMPLETED
File Name : Customer_Reviews_Dirty.csv
Rows      : 1181
Columns   : 6

Customer_Reviews_Dirty.csv exported successfully.
